# Bank Marketing — Categorical Feature Engineering


## Introduction

Marketing is one of the central pillars of any business — it drives revenue growth and helps an organization expand. In marketing, the goal is to reach the audience most likely to want a given product, thereby increasing sales.

Intelligent marketing is one of many applications of AI and machine learning in business, built on a simple principle:

> *"Show the product to the people who are most likely to buy it."*

The dataset used here comes from a bank's telemarketing campaign for a long-term deposit product. A marketing campaign is a coordinated set of activities aimed at raising customer awareness of a product; bank marketing specialists use a variety of methods to raise awareness of banking services.

We're given detailed information about the bank's customers, along with the outcomes of previous marketing campaigns. The goal is to predict, ahead of time, which customers are likely to subscribe to the new long-term deposit product being promoted in the current campaign.

This notebook focuses on the **preprocessing / feature-engineering** stage — turning raw categorical fields into model-ready numeric features. A lightweight AutoML baseline is included so the impact of the preprocessing can be measured directly, before and after.


## Dataset

Each row corresponds to one customer contacted as part of the bank's long-term deposit campaign.

| Column | Description |
|---|---|
| `age` | Customer's age |
| `job` | Customer's occupation |
| `marital` | Marital status |
| `education` | Education level |
| `default` | Whether the customer has credit in default* |
| `housing` | Whether the customer has a housing loan |
| `loan` | Whether the customer has a personal loan |
| `is_telephone_contact` | Whether the last contact was made via landline |
| `month` | Month of the last contact |
| `day_of_week` | Day of the week of the last contact |
| `duration` | Duration (seconds) of the last contact |
| `campaign` | Number of contacts made during this campaign for this client |
| `pdays` | Days since the client was last contacted in a previous campaign (999 = never contacted) |
| `previous` | Number of contacts made before this campaign |
| `poutcome` | Outcome of the previous marketing campaign |
| `y` | Outcome of the current campaign (**target**) |

*`default`: 0 = no default, 1 = client defaulted on their debt.

The dataset has **16 columns and 41,188 rows**. Missing values are encoded as the string `"unknown"` — wherever `unknown` appears, it should be treated as `NaN`. Imputing these values is itself part of the exercise below.


### Encoding reference

To convert categorical values into numbers, the following lookup tables are used throughout this notebook:

| Value in data | Encoded as |
| :---: | :---: |
| `yes` | `1` |
| `no` | `0` |

| Value in data | Encoded as |
| :---: | :---: |
| `illiterate` | `0` |
| `basic.4y` | `1` |
| `basic.6y` | `2` |
| `basic.9y` | `3` |
| `high.school` | `4` |
| `professional.course` | `5` |
| `university.degree` | `6` |

| Value in data | Encoded as |
| :---: | :---: |
| `mon` | `0` |
| `tue` | `1` |
| `wed` | `2` |
| `thu` | `3` |
| `fri` | `4` |
| `sat` | `5` |
| `sun` | `6` |

| Value in data | Encoded as |
| :---: | :---: |
| `jan` | `0` |
| `feb` | `1` |
| `mar` | `2` |
| `apr` | `3` |
| `may` | `4` |
| `jun` | `5` |
| `jul` | `6` |
| `aug` | `7` |
| `sep` | `8` |
| `oct` | `9` |
| `nov` | `10` |
| `dec` | `11` |

| Value in data | Encoded as |
| :---: | :---: |
| `failure` | `-1` |
| `nonexistent` | `0` |
| `success` | `1` |


Import the required libraries and load the dataset.


In [2]:
#%pip install flaml
%pip install "flaml[automl]"

import pandas as pd
from flaml import AutoML
from sklearn.metrics import f1_score

  Using cached flaml-2.6.0-py3-none-any.whl.metadata (12 kB)
  Using cached lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl.metadata (17 kB)
  Using cached xgboost-2.1.4-py3-none-manylinux_2_28_x86_64.whl.metadata (2.1 kB)
Using cached flaml-2.6.0-py3-none-any.whl (349 kB)
Using cached xgboost-2.1.4-py3-none-manylinux_2_28_x86_64.whl (223.6 MB)
Using cached lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl (3.6 MB)
  Attempting uninstall: xgboost━━━━━━━━━━━━━━━━━ 0/3 [flaml]
    Found existing installation: xgboost 3.2.00m 0/3 [flaml]
    Uninstalling xgboost-3.2.0:━━━━━━━━━━━━━ 0/3 [flaml]
      Successfully uninstalled xgboost-3.2.0 0/3 [flaml]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [lightgbm]2/3 [lightgbm]
Note: you may need to restart the kernel to use updated packages.


In [3]:
df = pd.read_csv('data/bank-additional-full.csv', na_values='unknown')
df.head()


,age,job,marital,education,default,housing,loan,is_telephone_contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,y
0,56,housemaid,married,basic.4y,no,no,no,yes,may,mon,261,1,999,0,nonexistent,no
1,57,services,married,high.school,NaN,no,no,yes,may,mon,149,1,999,0,nonexistent,no
2,37,services,married,high.school,no,yes,no,yes,may,mon,226,1,999,0,nonexistent,no
3,40,admin.,married,basic.6y,no,no,no,yes,may,mon,151,1,999,0,nonexistent,no
4,56,services,married,high.school,no,no,yes,yes,may,mon,307,1,999,0,nonexistent,no


### Baseline model

To appreciate the value of the preprocessing step, we first train a model directly on the raw, unprocessed data. Running the cell below trains a model and reports its performance (this may take a couple of minutes).

If the cell fails, it's likely one of the following:
- The `flaml[automl]` package isn't installed.
- There was an issue loading the dataset.


In [4]:
target_variable = 'y'
df_train = df.sample(frac=.7, random_state=313)

model = AutoML(task='classification', time_budget=120, verbose=0, estimator_list=['lgbm', 'rf'])
model.fit(df_train.drop(target_variable, axis=1), df_train[target_variable])

df_test = df.drop(df_train.index)
y_pred = model.predict(df_test.drop(target_variable, axis=1))

f1score = f1_score(df_test[target_variable], y_pred, pos_label='yes')*100
print(f'performance of model is {f1score} %')

performance of model is 53.03867403314917 %


## Step 1 — Impute Missing Categorical Values

The categorical columns of this dataframe contain missing values. For this step:

1. Store the names of all categorical columns in a list called `categorical_cols` (the target variable should **not** be included).
2. For every column in `categorical_cols`, replace missing values with that column's most frequent value (mode). Store the result back into `df`, which should now have no missing values.


In [16]:
categorical_cols = [
    'job',
    'marital',
    'education',
    'default',
    'housing',
    'loan',
    'is_telephone_contact',
    'month',
    'day_of_week',
    'poutcome'
]

df = df.copy()
df = df.replace("unknown", pd.NA)

for col in categorical_cols:
    most_frequent = df[col].mode()[0]
    df[col] = df[col].fillna(most_frequent)
    
df.head()

,age,job,marital,education,default,housing,loan,is_telephone_contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,y
0,56,housemaid,married,basic.4y,no,no,no,yes,may,mon,261,1,999,0,nonexistent,no
1,57,services,married,high.school,no,no,no,yes,may,mon,149,1,999,0,nonexistent,no
2,37,services,married,high.school,no,yes,no,yes,may,mon,226,1,999,0,nonexistent,no
3,40,admin.,married,basic.6y,no,no,no,yes,may,mon,151,1,999,0,nonexistent,no
4,56,services,married,high.school,no,no,yes,yes,may,mon,307,1,999,0,nonexistent,no


## Step 2 — Classify Columns by Type

Build four lists that classify the columns of `df` by data type:

- `continuous_cols` — columns with continuous values.
- `binary_cols` — columns with binary (yes/no) values.
- `nominal_cols` — columns with nominal (unordered categorical) values.
- `ordinal_cols` — columns with ordinal (ordered categorical) values.

All four variables must be of type `list`.


In [10]:
continuous_cols = ['age', 'duration', 'campaign', 'pdays', 'previous']

binary_cols = ['default', 'housing', 'loan', 'is_telephone_contact']

nominal_cols = ['job', 'marital']

ordinal_cols = ['education', 'month', 'day_of_week', 'poutcome']


## Step 3 — Encode Binary Columns

Build `binary_df`: a dataframe containing only the binary columns of `df`, with all values converted to numbers (`yes` → `1`, `no` → `0`).


In [11]:
# TO-DO: create `binary_df` and replace all `object` values with numeric ones

binary_df = df[binary_cols].copy()

for col in binary_cols:
    binary_df[col] = binary_df[col].map({'yes': 1, 'no': 0})

binary_df.head()



,default,housing,loan,is_telephone_contact
0,0,0,0,1
1,0,0,0,1
2,0,1,0,1
3,0,0,0,1
4,0,0,1,1


## Step 4 — One-Hot Encode Nominal Columns

Build `nominal_df`: a dataframe containing the one-hot encoded version of every nominal column in `nominal_cols`. Since nominal features have no inherent order, they're one-hot encoded, with each resulting column named `<original_column>_<value>`.

For example, given a hypothetical column `imaginary_col`:

| `imaginary_col` |
| :---: |
| `val1` |
| `val1` |
| `val2` |
| `val3` |
| `val2` |
| `val1` |
| `val3` |

...the encoded result should look like:

| `imaginary_col_val1` | `imaginary_col_val2` | `imaginary_col_val3` |
| :---: | :---: | :---: |
| `1` | `0` | `0` |
| `1` | `0` | `0` |
| `0` | `1` | `0` |
| `0` | `0` | `1` |
| `0` | `1` | `0` |
| `1` | `0` | `0` |
| `0` | `0` | `1` |

Repeat this for every nominal column, then concatenate the results into `nominal_df`. Note that `nominal_df` should **not** contain the original nominal columns — only their one-hot encoded versions.


In [12]:
# TO-DO: create `nominal_df` according to above explanations

nominal_subset = df[nominal_cols].copy()
nominal_df = pd.get_dummies(nominal_subset, prefix=nominal_cols)
nominal_df = nominal_df.astype(int)

nominal_df.head()

,job_admin.,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,marital_divorced,marital_married,marital_single
0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
1,0,0,0,0,0,0,0,1,0,0,0,0,1,0
2,0,0,0,0,0,0,0,1,0,0,0,0,1,0
3,1,0,0,0,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,0,0,0,1,0,0,0,0,1,0


## Step 5 — Encode Ordinal Columns

Build `ordinal_df`: a dataframe containing only the ordinal columns of `df`, with all values converted to numbers using the [encoding reference](#Encoding-reference) tables above.


In [18]:

ordinal_df = df[ordinal_cols].copy()

education_map = {
    'illiterate': 0,
    'basic.4y': 1,
    'basic.6y': 2,
    'basic.9y': 3,
    'high.school': 4,
    'professional.course': 5,
    'university.degree': 6
}

day_map = {
    'mon': 0,
    'tue': 1,
    'wed': 2,
    'thu': 3,
    'fri': 4,
    'sat': 5,
    'sun': 6
}

month_map = {
    'jan': 0,
    'feb': 1,
    'mar': 2,
    'apr': 3,
    'may': 4,
    'jun': 5,
    'jul': 6,
    'aug': 7,
    'sep': 8,
    'oct': 9,
    'nov': 10,
    'dec': 11
}

poutcome_map = {
    'failure': -1,
    'nonexistent': 0,
    'success': 1
}

ordinal_df['education'] = ordinal_df['education'].map(education_map)
ordinal_df['day_of_week'] = ordinal_df['day_of_week'].map(day_map)
ordinal_df['month'] = ordinal_df['month'].map(month_map)
ordinal_df['poutcome'] = ordinal_df['poutcome'].map(poutcome_map)

ordinal_df.head()

,education,month,day_of_week,poutcome
0,1,4,0,0
1,4,4,0,0
2,4,4,0,0
3,2,4,0,0
4,4,4,0,0


## Model Evaluation After Preprocessing

Now that preprocessing is complete, we retrain the model on the cleaned dataset to measure the impact of feature engineering on model performance.

If this cell fails, it's likely one of the following:
- The `flaml[automl]` package isn't installed.
- One of the preprocessing steps above wasn't implemented correctly.


In [15]:
target_variable = 'y'
new_df = df[continuous_cols].join([binary_df, nominal_df, ordinal_df])

model = AutoML(task='classification', time_budget=120, verbose=0, estimator_list=['lgbm', 'rf'])
model.fit(new_df, df[target_variable])

df_test = new_df.join(df.y).sample(frac=0.3, random_state=313)
y_pred = model.predict(df_test.drop(target_variable, axis=1))

f1score = f1_score(df_test[target_variable], y_pred, pos_label='yes')*100
print(f'performance of model is {f1score} %')

performance of model is 66.0764587525151 %


You can experiment with additional preprocessing steps to further improve model performance.


---

## Summary

### Preprocessing
Columns were first inspected and grouped into four types: continuous, binary, nominal, and ordinal.

`unknown` values were treated as missing and, for categorical columns, imputed with the column's mode.

To prepare the data for modeling:
- **Binary** features (`housing`, `loan`, `default`, `is_telephone_contact`) were mapped to `0`/`1`.
- **Nominal** features (`job`, `marital`) were one-hot encoded.
- **Ordinal** features (`education`, `month`, `day_of_week`, `poutcome`) were mapped to ordered numeric codes.

### Modeling
A [FLAML](https://github.com/microsoft/FLAML) AutoML search was used, evaluating LightGBM and Random Forest. The model was trained on 70% of the data and evaluated on the remaining 30%, using F1-score as the evaluation metric.

### Results

| Model | F1-score |
|---|---|
| Baseline (raw data) | see notebook output |
| **After feature engineering** | **65.44%** |

### Conclusion
Careful, type-aware encoding of categorical features (rather than naive label encoding) has a measurable, positive effect on downstream model performance.
